In [1]:
import os
import re
import warnings
import threading
from collections import Counter

import pandas as pd
import numpy as np
import scipy.sparse as sp
import nltk
from joblib import Parallel, delayed
import joblib
import json

import xgboost as xgb
import lightgbm as lgb
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import paired_cosine_distances
from sklearn.metrics import f1_score, classification_report

warnings.filterwarnings('ignore')

# Step 0: Load Resources
nltk.download(['punkt', 'punkt_tab', 'averaged_perceptron_tagger_eng'], quiet=True)

# Step 1: Load Data
DATA_DIR = "/kaggle/input/datasets/dongmyungpark/nlu-av/" 

def load_data(directory):
    train_path = os.path.join(directory, "train.csv")
    dev_path = os.path.join(directory, "dev.csv")
    
    if not os.path.exists(train_path):
        raise FileNotFoundError(f"Path not found: {train_path}")
        
    train = pd.read_csv(train_path)
    dev = pd.read_csv(dev_path)
    
    print(f"✅ Data Loaded | Train: {len(train):,} | Dev: {len(dev):,}")
    return train, dev

train_df, dev_df = load_data(DATA_DIR)
train_df.head()

print(train_df.shape, dev_df.shape)
print(train_df['label'].value_counts())
print(train_df.isnull().sum())

✅ Data Loaded | Train: 27,643 | Dev: 5,993
(27643, 3) (5993, 3)
label
0.0    13950
1.0    13693
Name: count, dtype: int64
text_1    0
text_2    0
label     0
dtype: int64


In [2]:
# 0. Text regularization
def normalize_text(text):
    text = str(text)
    text = re.sub(r'[\w.+-]+@[\w.]+\.[a-z]{2,}', ' ', text)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'\d{1,2}[/-]\d{1,2}[/-]\d{2,4}', ' ', text)
    text = re.sub(r'\d{1,2}:\d{2}(:\d{2})?', ' ', text)
    text = re.sub(r'\(?\d{3}\)?[-.\s]\d{3}[-.\s]\d{4}', ' ', text)
    text = re.sub(r'(.)\1{3,}', r'\1\1', text)
    text = re.sub(r'([!?.]){2,}', r'\1', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# 1. Features
def process_text_stylometric(text):
    text = str(text)
    words = text.split()
    ln = len(words)

    if ln < 5:
        return np.zeros(24, dtype='float32')

    sentences = [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]
    word_lens = [len(w) for w in words]
    sent_lens = [len(s.split()) for s in sentences] if sentences else [ln]

    return np.array([
        ln,
        np.mean(word_lens),
        np.mean(sent_lens),
        len(set(words)) / ln, 
        
        np.std(word_lens),
        np.std(sent_lens),
        np.max(word_lens),
        np.median(sent_lens),

        text.count(',') / ln,
        text.count('.') / ln,
        text.count('!') / ln,
        text.count('?') / ln,
        text.count(';') / ln,
        text.count(':') / ln,
        text.count('"') / ln,
        text.count("'") / ln,

        sum(w.isupper() for w in words) / ln,
        sum(len(w) > 6 for w in words) / ln,
        sum(len(w) <= 3 for w in words) / ln,
        sum(w[0].isupper() for w in words if w) / ln,

        sum(w.endswith('ly') for w in words) / ln,
        sum(w.endswith(('tion', 'sion')) for w in words) / ln,
        sum(w.endswith('ing') for w in words) / ln,
        sum(w.endswith('ed') for w in words) / ln,
    ], dtype='float32')


def fast_feature_extraction(df):
    texts_1 = df['text_1'].apply(normalize_text).values
    texts_2 = df['text_2'].apply(normalize_text).values
    all_texts = np.concatenate([texts_1, texts_2])
    mid = len(df)

    style_results = Parallel(n_jobs=-1, batch_size=1000)(
        delayed(process_text_stylometric)(t) for t in all_texts
    )
    s1 = np.array(style_results[:mid])
    s2 = np.array(style_results[mid:])

    diff  = np.abs(s1 - s2)
    ratio = np.minimum(s1, s2) / (np.maximum(s1, s2) + 1e-9)
    mean  = (s1 + s2) / 2

    combined = np.hstack([diff, ratio, mean]).astype('float32')
    return sp.csr_matrix(combined, dtype='float32'), texts_1, texts_2

# 2. TF-IDF Fitting
FUNCTION_WORDS = [
    'the', 'a', 'an', 'and', 'or', 'but', 'if', 'in', 'on', 'at', 'to', 'for',
    'of', 'with', 'by', 'from', 'as', 'is', 'was', 'are', 'were', 'be', 'been',
    'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would', 'could', 'should',
    'may', 'might', 'shall', 'that', 'which', 'who', 'whom', 'this', 'these', 'those',
    'i', 'we', 'you', 'he', 'she', 'they', 'it', 'my', 'your', 'his', 'her',
    'however', 'therefore', 'moreover', 'furthermore', 'although', 'though',
    'because', 'since', 'while', 'when', 'where', 'what', 'how', 'why', 'just', 'also',
    'very', 'really', 'quite', 'rather', 'so', 'too', 'even', 'still', 'already',
    'not', 'only', 'then', 'here', 'there', 'now', 'never', 'always', 'well',
    'me', 'us', 'him', 'them', 'its', 'our', 'their', 'about', 'into', 'through',
    'over', 'under', 'before', 'after', 'cannot', 'without',

    'of the', 'in the', 'to the', 'on the', 'for the', 'at the', 'by the',
    'with the', 'from the', 'and the', 'of a', 'in a', 'to a', 'for a', 'with a',

    'it is', 'it was', 'there is', 'there are', 'there was', 'there were',
    'to be', 'will be', 'would be', 'could be', 'should be', 'may be',
    'is a', 'is the', 'was a', 'was the', 'are the', 'were the',

    'do not', 'does not', 'did not', 'will not', 'would not',
    'could not', 'should not', 'have not', 'has not', 'had not',
    'is not', 'are not', 'was not', 'were not',

    'i am', 'i was', 'i have', 'i had', 'i will', 'i would', 'i think',
    'i know', 'i do', 'i can', 'i could', 'i feel', 'i want', 'i need',
    'we are', 'we have', 'we will', 'we were', 'we had',
    'you are', 'you have', 'you will', 'you were', 'you can',
    'he was', 'he is', 'he had', 'he would', 'he said',
    'she was', 'she is', 'she had', 'she would', 'she said',
    'they are', 'they were', 'they have', 'they had', 'they will',

    'to me', 'for me', 'of it', 'and i', 'but i', 'so i', 'that i',

    'as well', 'as a', 'such as', 'as the', 'up to', 'out of',
    'in order', 'in fact', 'in addition', 'at least', 'at all',
    'of course', 'as if', 'even if', 'even though',
    'so that', 'in that', 'now that', 'given that',
    'due to', 'based on', 'rather than'
]

print("Step 1: Normalizing texts...")
train_df['text_1_clean'] = train_df['text_1'].apply(normalize_text)
train_df['text_2_clean'] = train_df['text_2'].apply(normalize_text)
dev_df['text_1_clean']   = dev_df['text_1'].apply(normalize_text)
dev_df['text_2_clean']   = dev_df['text_2'].apply(normalize_text)

print("Step 2: TF-IDF Fitting...")
all_txt = pd.concat([
    train_df['text_1_clean'], train_df['text_2_clean'],
    dev_df['text_1_clean'],   dev_df['text_2_clean']
])

v_w  = TfidfVectorizer(ngram_range=(1, 2), max_features=7000, dtype=np.float32)
v_c  = TfidfVectorizer(analyzer='char', ngram_range=(2, 4), max_features=7000, dtype=np.float32)
v_fw = TfidfVectorizer(vocabulary=FUNCTION_WORDS, ngram_range=(1, 2), dtype=np.float32)

v_w.fit(all_txt)
v_c.fit(all_txt)
v_fw.fit(all_txt)

print("Step 3: Style Feature Extraction...")
train_s, _, _  = fast_feature_extraction(train_df)
dev_s,   _, _  = fast_feature_extraction(dev_df)


# 3. Matrix
def build_matrix(df, s):
    w1  = v_w.transform(df['text_1_clean'])
    c1  = v_c.transform(df['text_1_clean'])
    fw1 = v_fw.transform(df['text_1_clean'])

    w2  = v_w.transform(df['text_2_clean'])
    c2  = v_c.transform(df['text_2_clean'])
    fw2 = v_fw.transform(df['text_2_clean'])

    x1 = sp.hstack([w1, c1])
    x2 = sp.hstack([w2, c2])

    diff    = abs(x1 - x2)
    prod    = x1.multiply(x2)

    fw_diff = abs(fw1 - fw2)
    fw_prod = fw1.multiply(fw2)

    cos    = paired_cosine_distances(x1,  x2).reshape(-1, 1).astype('float32')
    fw_cos = paired_cosine_distances(fw1, fw2).reshape(-1, 1).astype('float32')

    return sp.hstack(
        [diff, prod, fw_diff, fw_prod, s, cos, fw_cos],
        format='csr', dtype='float32'
    )

print("Step 4: Matrix Consolidation...")
train_features = build_matrix(train_df, train_s)
dev_features   = build_matrix(dev_df,   dev_s)
print(f"Completed! Shape: {train_features.shape}")

Step 1: Normalizing texts...
Step 2: TF-IDF Fitting...
Step 3: Style Feature Extraction...
Step 4: Matrix Consolidation...
Completed! Shape: (27643, 28512)


In [3]:
results = {}
models = {}

def train_xgb():
    print("🚀 XGBoost: GPU 0 Start")
    model = xgb.XGBClassifier(
        n_estimators=7000,
        learning_rate=0.01,
        max_depth=7,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=2,
        tree_method='hist',
        device='cuda:0',
        max_bin=128,
        random_state=10879360,
        early_stopping_rounds=200,
        eval_metric='logloss'
    )
    model.fit(
        train_features, train_df['label'],
        eval_set=[(dev_features, dev_df['label'])],
        verbose=100
    )
    results['xgb'] = model.predict_proba(dev_features)[:, 1]
    models['xgb'] = model
    print("✅ XGBoost: Done")

def train_lgb():
    print("🥊 LightGBM: GPU 1 Start")
    model = lgb.LGBMClassifier(
        n_estimators=2000,
        learning_rate=0.01,
        num_leaves=127,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        random_state=10879360,
        device='gpu',
        gpu_platform_id=0,
        gpu_device_id=1,
        max_bin=127
    )
    model.fit(
        train_features, train_df['label'],
        eval_set=[(dev_features, dev_df['label'])],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(period=100)
        ]
    )
    results['lgb'] = model.predict_proba(dev_features)[:, 1]
    models['lgb'] = model
    print("✅ LightGBM: Done")

t1 = threading.Thread(target=train_xgb)
t2 = threading.Thread(target=train_lgb)

t1.start(); t2.start()
t1.join(); t2.join()

probs_xgb = results['xgb']
probs_lgb = results['lgb']

print("\n🏁 Dual-GPU Training Complete.")

🚀 XGBoost: GPU 0 Start
🥊 LightGBM: GPU 1 Start
[0]	validation_0-logloss:0.69175
[LightGBM] [Info] Number of positive: 13693, number of negative: 13950
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 1994351
[LightGBM] [Info] Number of data points in the train set: 27643, number of used features: 22080
[LightGBM] [Info] Using requested OpenCL platform 0 device 1
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 975 dense feature groups (25.73 MB) transferred to GPU in 0.030106 secs. 1 sparse feature groups
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.495351 -> initscore=-0.018595
[LightGBM] [Info] Start training from score -0.018595
[100]	validation_0-logloss:0.60453
[200]	validation_0-logloss:0.56893
[300]	validation_0-logloss:0.54758
[400]	validation_0-logloss:0.53346
[500]	validation_0-logloss:0.52375
[600]	validation_0-logloss:0.51567
[700]	validation_0-logloss:0.50983
[800]	validation_0-logloss:0.50508
[900]	validation_0-logloss:0.50113
[100]	valid_0's binary_logloss: 0.568013
[1000]	validation_0-logloss:0.49770
[1100]	validation_0-logloss:0.49461
[1200]	validation_0-logloss:0.49188
[1300]	validation_0-logloss:0.48943
[1400]	validation_0-logloss:0.48698
[1500]	validation_0-logloss:0.48492
[1600]	validation_0-logloss:0.48322
[1700]	validation_0-logloss:0.48119
[1800]	vali

[12:18:45] WARNING: /workspace/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.



✅ XGBoost: Done
[700]	valid_0's binary_logloss: 0.463167
[800]	valid_0's binary_logloss: 0.458511
[900]	valid_0's binary_logloss: 0.454622
[1000]	valid_0's binary_logloss: 0.451613
[1100]	valid_0's binary_logloss: 0.449054
[1200]	valid_0's binary_logloss: 0.447105
[1300]	valid_0's binary_logloss: 0.445439
[1400]	valid_0's binary_logloss: 0.444468
[1500]	valid_0's binary_logloss: 0.444028
[1600]	valid_0's binary_logloss: 0.443647
[1700]	valid_0's binary_logloss: 0.443192
[1800]	valid_0's binary_logloss: 0.443317
✅ LightGBM: Done

🏁 Dual-GPU Training Complete.


In [4]:
# Feature Importance Analysis
style_cols = [
    'ln', 'mean_word_len', 'mean_sent_len', 'ttr',
    'std_word_len', 'std_sent_len', 'max_word_len', 'median_sent_len',
    'comma_r', 'period_r', 'exclaim_r', 'question_r',
    'semicolon_r', 'colon_r', 'quote_r', 'apos_r',
    'upper_r', 'long_word_r', 'short_word_r', 'cap_r',
    'adv_r', 'noun_r', 'ing_r', 'ed_r'
]

feature_names = (
    [f'diff_w_{v}'   for v in v_w.get_feature_names_out()] +
    [f'prod_w_{v}'   for v in v_w.get_feature_names_out()] +
    [f'diff_c_{v}'   for v in v_c.get_feature_names_out()] +
    [f'prod_c_{v}'   for v in v_c.get_feature_names_out()] +
    [f'fw_diff_{v}'  for v in v_fw.get_feature_names_out()] +
    [f'fw_prod_{v}'  for v in v_fw.get_feature_names_out()] +
    [f'style_diff_{c}'  for c in style_cols] +
    [f'style_ratio_{c}' for c in style_cols] +
    [f'style_mean_{c}'  for c in style_cols] +
    ['cos', 'fw_cos']
)

n_features = train_features.shape[1]
if len(feature_names) != n_features:
    print(f"⚠️ Feature name mismatch ({len(feature_names)} vs {n_features}) — falling back to indices")
    feature_names = [str(i) for i in range(n_features)]

def get_group(name):
    if name.startswith('diff_w'):      return 'word_diff'
    if name.startswith('prod_w'):      return 'word_prod'
    if name.startswith('diff_c'):      return 'char_diff'
    if name.startswith('prod_c'):      return 'char_prod'
    if name.startswith('fw_diff'):     return 'fw_diff'
    if name.startswith('fw_prod'):     return 'fw_prod'
    if name.startswith('style_diff'):  return 'style_diff'
    if name.startswith('style_ratio'): return 'style_ratio'
    if name.startswith('style_mean'):  return 'style_mean'
    if name in ('cos', 'fw_cos'):      return 'cosine'
    return 'other'

for model_name in ['xgb', 'lgb']:
    imp = models[model_name].feature_importances_
    fi = pd.DataFrame({'feature': feature_names, 'importance': imp})

    fi['group'] = fi['feature'].apply(get_group)
    group_sum = (fi.groupby('group')['importance']
                   .sum()
                   .sort_values(ascending=False)
                   .reset_index())
    group_sum['pct'] = (group_sum['importance'] / group_sum['importance'].sum() * 100).round(1)

    print(f"\n{'='*45}")
    print(f"  {model_name.upper()} Feature Importance by Group")
    print(f"{'='*45}")
    print(group_sum.to_string(index=False))

    print(f"\n  {model_name.upper()} Top 20 Individual Features")
    print(f"{'-'*45}")
    print(fi.nlargest(20, 'importance')[['feature', 'importance']].to_string(index=False))


  XGB Feature Importance by Group
      group  importance       pct
  word_prod    0.591880 59.200001
  word_diff    0.183968 18.400000
  char_prod    0.176148 17.600000
  char_diff    0.020940  2.100000
    fw_diff    0.012886  1.300000
    fw_prod    0.004747  0.500000
style_ratio    0.003020  0.300000
 style_mean    0.002943  0.300000
 style_diff    0.002472  0.200000
     cosine    0.000997  0.100000

  XGB Top 20 Individual Features
---------------------------------------------
            feature  importance
         prod_c_kay    0.003441
        prod_c_ kay    0.002148
diff_w_smith street    0.001885
    prod_w_look for    0.001794
   diff_w_street eb    0.001761
         diff_c_u/e    0.001495
  diff_w_start date    0.001486
 diff_w_77002 phone    0.001386
     prod_w_approve    0.001357
         diff_c_hot    0.001227
   diff_w_hourahead    0.001174
        prod_c_ vin    0.001054
    diff_w_kaminski    0.001025
       prod_w_vince    0.001020
      prod_w_chance    0.001008

In [5]:
print("\n--- Individual Model Scores ---")

# 1. Evaluate XGBoost
best_f1_xgb, best_th_xgb = 0, 0
for th in np.arange(0.2, 0.8, 0.01):
    preds = (probs_xgb >= th).astype(int)
    score = f1_score(dev_df['label'], preds)
    if score > best_f1_xgb:
        best_f1_xgb, best_th_xgb = score, th

print(f"🚀 XGBoost Max F1: {best_f1_xgb:.4f} (Threshold: {best_th_xgb:.2f})")

# 2. Evaluate LightGBM
best_f1_lgb, best_th_lgb = 0, 0
for th in np.arange(0.2, 0.8, 0.01):
    preds = (probs_lgb >= th).astype(int)
    score = f1_score(dev_df['label'], preds)
    if score > best_f1_lgb:
        best_f1_lgb, best_th_lgb = score, th

print(f"🥊 LightGBM Max F1: {best_f1_lgb:.4f} (Threshold: {best_th_lgb:.2f})")
print("-" * 35)


--- Individual Model Scores ---
🚀 XGBoost Max F1: 0.7777 (Threshold: 0.38)
🥊 LightGBM Max F1: 0.7819 (Threshold: 0.38)
-----------------------------------


In [6]:
print("🧪 Searching for the Golden Ratio (XGB vs LGBM)...")

best_overall_f1 = 0
best_weight = 0
best_final_thresh = 0

for w in np.arange(0.2, 0.81, 0.01):
    curr_probs = (probs_xgb * w) + (probs_lgb * (1 - w))
    
    for thresh in np.arange(0.2, 0.8, 0.01):
        preds = (curr_probs >= thresh).astype(int)
        score = f1_score(dev_df['label'], preds)
        
        if score > best_overall_f1:
            best_overall_f1 = score
            best_weight = w
            best_final_thresh = thresh

print("\n" + "="*40)
print(f"🎊 SEARCH COMPLETED! 🎊")
print(f"🥇 Best XGB Weight: {best_weight:.2f}")
print(f"🥈 Best LGBM Weight: {1-best_weight:.2f}")
print(f"🥉 Best Threshold: {best_final_thresh:.2f}")
print(f"🚀 MAX F1 SCORE: {best_overall_f1:.4f}")
print("="*40)

final_probs = (probs_xgb * best_weight) + (probs_lgb * (1 - best_weight))
final_preds = (final_probs >= best_final_thresh).astype(int)
print("\n--- Ultimate Ensemble Classification Report ---")
print(classification_report(dev_df['label'], final_preds))

🧪 Searching for the Golden Ratio (XGB vs LGBM)...

🎊 SEARCH COMPLETED! 🎊
🥇 Best XGB Weight: 0.43
🥈 Best LGBM Weight: 0.57
🥉 Best Threshold: 0.40
🚀 MAX F1 SCORE: 0.7826

--- Ultimate Ensemble Classification Report ---
              precision    recall  f1-score   support

           0       0.80      0.70      0.74      2937
           1       0.74      0.83      0.78      3056

    accuracy                           0.76      5993
   macro avg       0.77      0.76      0.76      5993
weighted avg       0.77      0.76      0.76      5993



In [7]:
OUTPUT_DIR = "/kaggle/working/models"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save models
models['xgb'].save_model(os.path.join(OUTPUT_DIR, "xgb_model.json"))
joblib.dump(models['lgb'], os.path.join(OUTPUT_DIR, "lgb_model.pkl"))

# Save vectorizers
joblib.dump(v_w,  os.path.join(OUTPUT_DIR, "tfidf_word.pkl"))
joblib.dump(v_c,  os.path.join(OUTPUT_DIR, "tfidf_char.pkl"))
joblib.dump(v_fw, os.path.join(OUTPUT_DIR, "tfidf_fw.pkl"))

# Save ensemble config
ensemble_config = {
    "xgb_weight":    round(best_weight, 2),
    "lgb_weight":    round(1 - best_weight, 2),
    "threshold":     round(best_final_thresh, 2),
    "best_f1":       round(best_overall_f1, 4),
}
with open(os.path.join(OUTPUT_DIR, "ensemble_config.json"), "w") as f:
    json.dump(ensemble_config, f, indent=2)

print("✅ All models saved.")
print(f"   📦 XGBoost    → xgb_model.json")
print(f"   📦 LightGBM   → lgb_model.pkl")
print(f"   📦 Vectorizers → tfidf_word/char/fw.pkl")
print(f"   📦 Ensemble   → ensemble_config.json")
print(f"\n   XGB weight : {ensemble_config['xgb_weight']}")
print(f"   LGB weight : {ensemble_config['lgb_weight']}")
print(f"   Threshold  : {ensemble_config['threshold']}")
print(f"   Best F1    : {ensemble_config['best_f1']}")

✅ All models saved.
   📦 XGBoost    → xgb_model.json
   📦 LightGBM   → lgb_model.pkl
   📦 Vectorizers → tfidf_word/char/fw.pkl
   📦 Ensemble   → ensemble_config.json

   XGB weight : 0.43
   LGB weight : 0.57
   Threshold  : 0.4
   Best F1    : 0.7826
